# `n_reads` and the exposure-time inverses

Two related `Simulation` capabilities:

- **The exposure-time inverse** — `get_image_exptime_for_snr(snr)` answers *how long do I
  integrate* to reach a target SNR, inverting `get_image_snr`. (The older analytic
  `get_exptime_for_snr` is deprecated in favour of this PSF-aware path.)

- **`n_reads`** — the number of detector frames spanning the *total* exposure `t`, so each
  frame integrates `t / n_reads`. The read noise is paid **per frame**, so more reads cost
  SNR at a fixed total time. The same per-frame split lowers the peak charge in each frame,
  which can pull a bright star back under full well.

`n_reads` is a settable parameter (default 1, validated `>= 1`): set it with
`sim.update(n_reads=N)` or pass `n_reads=` per call.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import wcc_etc


def make_sim(mag, sensor="sony:r"):
    """A G5V source of the given magnitude on the standard zodi background."""
    scene = wcc_etc.get_scene(
        name="G5V",
        mag=mag,
        background="zodi",
        bandpass="johnson_r",
        background_prop={"bandpass": "johnson_r", "mag": 22.5},
    )
    return wcc_etc.Simulation.from_sensor_and_scene(sensor, scene)


wcc_etc.set_wcc_style()
sim = make_sim(20)

## 1. How long to reach a target SNR

`get_image_exptime_for_snr` returns a dict whose `'time_s'` is the answer, alongside the
aperture it solved for. Once a source is background limited the required time grows as
roughly SNR², and every magnitude of extra faintness costs another factor of ~2.5 in flux.

In [ ]:
targets = np.linspace(10, 200, 25)

fig, ax = plt.subplots(figsize=(7, 4.2))
for mag in (19, 20, 21):
    faint_sim = make_sim(mag)
    ax.plot(
        targets,
        [faint_sim.get_image_exptime_for_snr(s)["time_s"] for s in targets],
        label=f"r = {mag}",
    )

ax.set_xlabel("Target SNR")
ax.set_ylabel("Required exposure time [s]")
ax.set_title("Time to reach a target SNR (G5V, sony:r)")
ax.legend(fontsize=9)
plt.show()

res = sim.get_image_exptime_for_snr(100.0)
print(f"r = 20, SNR = 100 needs {res['time_s']:.1f} s")
print(f"  aperture radius   : {res['r_aper_mas']:.1f} mas")
print(f"  enclosed fraction : {res['enclosed_fraction']:.3f}")
print(f"  noise pixels      : {res['n_pix']}")

## 2. The cost of more reads

At a fixed total time the read noise is incurred once per frame, so the SNR falls with
`n_reads` — and conversely the time needed for a fixed target SNR grows. The effect is
largest for a faint, read-noise-affected source, so this uses r = 19.

At 60 s on the r = 19 source, going from 1 to 16 reads costs about **9%** of the SNR
(**192.7 → 175.4**). The penalty is real but modest here because the source is not yet
read-noise dominated; it grows as the source gets fainter.

In [ ]:
faint = make_sim(19)
reads = [1, 2, 4, 8, 16]
fixed_t, snr_goal = 60.0, 20.0

snr_vs_reads, t_vs_reads = [], []
for n in reads:
    faint.update(n_reads=n)
    snr_vs_reads.append(float(faint.get_snr(fixed_t)["snr"]))
    t_vs_reads.append(faint.get_image_exptime_for_snr(snr_goal)["time_s"])
faint.update(n_reads=1)

fig, (ax_snr, ax_t) = plt.subplots(1, 2, figsize=(11, 4.2))
ax_snr.plot(reads, snr_vs_reads, "o-")
ax_snr.axhline(snr_vs_reads[0], color="grey", ls=":", lw=1, label="n_reads = 1")
ax_snr.set_xlabel("n_reads")
ax_snr.set_ylabel(f"SNR at {fixed_t:.0f} s")
ax_snr.set_title("SNR at fixed total time")
ax_snr.legend(fontsize=9)

ax_t.plot(reads, t_vs_reads, "o-", color="C1")
ax_t.set_xlabel("n_reads")
ax_t.set_ylabel(f"Time for SNR = {snr_goal:.0f} [s]")
ax_t.set_title("Time for a fixed target SNR")

fig.suptitle("Read-noise penalty vs n_reads (G5V, r = 19)", y=1.02)
fig.tight_layout()
plt.show()

## 3. Reads as a saturation escape hatch

Saturation is evaluated on a single `t / n_reads` frame, so the per-frame peak falls as
`1 / n_reads`. A bright star that clips in one long frame can be brought back under the
ADC full scale by splitting the same total time across more reads.

An r = 17 star peaks at **~98,600 ADU** in a single 60 s frame, well past the 65,535
full scale. Splitting the same 60 s into just **2 reads** brings the per-frame peak back
under the ceiling.

In [ ]:
bright = make_sim(17)
total_t = 60.0
sat_reads = np.array([1, 2, 5, 10, 25, 50])

peaks = []
for n in sat_reads:
    bright.update(n_reads=int(n))
    peaks.append(bright.get_peak_pixel(total_t, units="adu").value)
peaks = np.array(peaks)

adc_max = bright.sensor.adc_max.value
below = sat_reads[peaks < adc_max]

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.loglog(sat_reads, peaks, "o-", label="per-frame peak pixel")
ax.axhline(adc_max, color="crimson", ls="--", label=f"adc_max = {adc_max:.0f} ADU")
if below.size:
    ax.axvline(
        below[0],
        color="k",
        ls=":",
        lw=1,
        label=f"clears full scale at n_reads = {below[0]}",
    )
ax.set_xlabel("n_reads")
ax.set_ylabel("Peak pixel per frame [ADU]")
ax.set_title(f"Splitting a {total_t:.0f} s exposure (G5V, r = 17, sony:r)")
ax.legend(fontsize=9)
plt.show()

## Summary

- `get_exptime_for_snr` (analytic) and `get_image_exptime_for_snr` (PSF-aware) invert the
  SNR calculation; the PSF-aware one also reports the aperture it solved for.
- `n_reads` splits the total exposure into frames and pays the read noise once per frame:
  at fixed time the SNR drops, and for a fixed target SNR the required time grows.
- Saturation is **per frame** (`t / n_reads`), so raising `n_reads` lowers the peak charge
  and can bring a clipping star back under full scale.
- `n_reads=1` is the single-frame default.